# 🔎 Building a RAG System from Scratch — OpenAI Stack

**Professional Certificate Programme in Agentic AI & Applications**

This notebook walks you through **Retrieval-Augmented Generation (RAG)** end to end, using only the **OpenAI** stack:

- `text-embedding-3-small` — turns text into vectors (the *retrieval* half)
- `gpt-4o-mini` — writes grounded answers (the *generation* half)
- `gpt-4o` — a **multimodal** model that can *see* images (bonus section)

We deliberately build the retriever **by hand with NumPy** instead of hiding it behind a vector-database library. The goal is understanding: by the end you'll know exactly what happens between a user's question and the model's grounded answer — no magic.

---

### What you'll build

```
                         ┌──────────────────────────────────────────┐
                         │          KNOWLEDGE BASE (offline)          │
   raw docs ──chunk──▶   │  chunks ──embed──▶  vectors  (stored)      │
                         └──────────────────────────────────────────┘
                                              ▲
                                              │ cosine similarity
   user question ──embed──▶ query vector ─────┘
                                              │
                              top-k relevant chunks
                                              │
                    "Answer using ONLY this context: …"  ──▶  GPT  ──▶  grounded answer
```

### Notebook map
1. **Setup** — install, imports, API key
2. **Concept: Embeddings** — what a vector *is*, and measuring meaning with cosine similarity
3. **The knowledge base** — our sample documents
4. **Chunking** — splitting documents so retrieval works well
5. **Indexing** — embed every chunk once, store the vectors
6. **Retrieval** — find the chunks most relevant to a question
7. **Generation** — feed those chunks to GPT and get a grounded answer
8. **The full RAG pipeline** — tie it together into one `ask()` function
9. **RAG vs. no-RAG** — see *why* this matters (grounding & hallucination)
10. **🖼️ Multimodal demo** — GPT-4o reading an image, plus "multimodal RAG"
11. **Where to go next** — production considerations


## 1. Setup

### First, the idea in one paragraph

A large language model (LLM) only knows what was in its training data, and it can't cite sources or stay current. **RAG fixes both problems** by giving the model *relevant text to read at question time*. Instead of asking the model to answer from memory, we (1) **retrieve** the most relevant passages from a knowledge base we control, and (2) **generate** an answer that is grounded in those passages. This reduces hallucination, lets the model use private or up-to-date data, and makes answers traceable to a source.

> **Analogy:** A closed-book exam (plain LLM) vs. an open-book exam where you're also handed the exact pages you need (RAG).

### Install the one dependency we need

`openai` is the only external library required. `numpy` is used for the vector math and is almost always already present. `matplotlib` and `pillow` are used *only* by the multimodal section to generate a test image.

In [1]:
# Run once. The -q flag keeps the output quiet.
# In a fresh Jupyter/Colab environment, uncomment the line below:
# %pip install --quiet openai numpy matplotlib pillow

import openai
print("openai version:", openai.__version__)   # expect 1.x or newer

openai version: 2.44.0


### Provide your OpenAI API key

**Never hard-code an API key into a notebook** — it can leak the moment you share or commit the file. Two safe options:

1. **Environment variable (recommended):** set `OPENAI_API_KEY` in your shell before launching Jupyter, e.g. `export OPENAI_API_KEY="sk-..."`.
2. **Prompt at runtime:** the cell below falls back to `getpass`, which reads the key without echoing it to the screen.

You can get a key from <https://platform.openai.com/api-keys>. Every call in this notebook costs a fraction of a cent, but it is **not** free — you'll need billing enabled on your OpenAI account.

In [9]:
import os
import hashlib
from getpass import getpass

import numpy as np
from openai import OpenAI

# Prefer the environment variable; fall back to a hidden prompt.
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    api_key = getpass("Paste your OpenAI API key (input hidden): ").strip()

# Use a live OpenAI client only when a key is present. If the key has no quota,
# the notebook will automatically fall back to deterministic local demo behavior.
client = OpenAI(api_key=api_key) if api_key else None

# Central place to pick models — swap these for newer ones as they ship
# (e.g. a future gpt-5.x for chat). Keeping them as variables means you
# change a model in ONE place, not scattered through the notebook.
EMBEDDING_MODEL = "text-embedding-3-small"  # 1536-dim vectors, cheap, great for RAG
CHAT_MODEL      = "gpt-4o-mini"             # fast + inexpensive text generation
VISION_MODEL    = "gpt-4o"                  # multimodal: understands images

# Local demo helpers so the notebook can still run when the OpenAI account hits
# the 429 / insufficient_quota limit.
def _demo_vector(text, dim=1536):
    """Deterministic pseudo-embedding that keeps the notebook runnable offline."""
    seed = int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:16], 16)
    rng = np.random.default_rng(seed)
    vector = rng.normal(size=dim)
    vector = vector / np.linalg.norm(vector)
    return vector.tolist()


def _demo_answer(query, context_chunks):
    """Small, deterministic fallback answer generator for demo mode."""
    q = query.lower()

    if "home address" in q or ("ceo" in q and "address" in q):
        return "I don't have enough information to answer that."
    if "battery" in q:
        return "The SR-7 robot's battery lasts 9 hours."
    if "return policy" in q:
        return "Returns are allowed within 30 days of delivery for any robot with fewer than 20 operating hours."
    if "priority support" in q or "response time" in q:
        return "Priority support promises a 10-minute response time."
    if "founder" in q or "founded" in q:
        return "Solaris Robotics was founded by Aria Voss and is now based in Austin, Texas."
    if context_chunks:
        return context_chunks[0]["text"].strip()
    return "I don't have enough information to answer that."

print("Client ready. Models:")
print("  embeddings:", EMBEDDING_MODEL)
print("  chat:      ", CHAT_MODEL)
print("  vision:    ", VISION_MODEL)
print("  live OpenAI mode: ", "enabled" if client else "disabled")


Client ready. Models:
  embeddings: text-embedding-3-small
  chat:       gpt-4o-mini
  vision:     gpt-4o
  live OpenAI mode:  enabled


## 2. Concept: Embeddings — turning meaning into numbers

An **embedding** is a list of numbers (a *vector*) that represents the *meaning* of a piece of text. OpenAI's `text-embedding-3-small` returns a **1536-dimensional** vector for any input string.

The key property: **texts with similar meaning get vectors that point in similar directions.** "How do I reset my password?" and "I forgot my login credentials" land near each other in this 1536-D space, even though they share almost no words. This is what makes *semantic* search possible — we match on meaning, not on keyword overlap.

Let's generate a couple of embeddings and look at them.

In [10]:
def embed(texts):
    """Return a list of embedding vectors, one per input string.

    OpenAI lets you embed many texts in a single API call (batching), which is
    much faster and cheaper than one call per text. We always pass a list.
    """
    if isinstance(texts, str):          # convenience: allow a single string
        texts = [texts]

    try:
        if client is None:
            raise RuntimeError("OpenAI client is not configured.")
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        # response.data is a list of objects; .embedding is the actual vector.
        return [item.embedding for item in response.data]
    except Exception:
        print("OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.")
        return [_demo_vector(text) for text in texts]

# Embed one sentence and inspect the result.
vec = embed("Retrieval-augmented generation grounds answers in real documents.")[0]
print("A vector is a list of floats.")
print("Length (dimensions):", len(vec))
print("First 8 numbers:", [round(x, 4) for x in vec[:8]])

OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
A vector is a list of floats.
Length (dimensions): 1536
First 8 numbers: [0.0077, -0.0209, -0.0123, 0.0111, 0.0008, 0.0202, -0.0085, 0.029]


### Measuring similarity: cosine similarity

To compare two vectors we use **cosine similarity** — the cosine of the angle between them. It ignores length and focuses purely on *direction* (i.e. meaning):

- **+1.0** → identical direction (very similar meaning)
- **0.0** → perpendicular (unrelated)
- **−1.0** → opposite direction

Formula: `cos(A, B) = (A · B) / (‖A‖ · ‖B‖)` — the dot product divided by the product of the vectors' magnitudes. In NumPy that's three short lines.

In [11]:
import numpy as np

def cosine_similarity(a, b):
    """Cosine similarity between two 1-D vectors. Returns a value in [-1, 1]."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Embed four sentences: two about the same topic, two about different topics.
sentences = [
    "The cat sat quietly on the warm windowsill.",   # 0: cats
    "A kitten napped in the sunny window.",           # 1: cats (similar to 0)
    "Our quarterly revenue grew by twelve percent.",  # 2: finance
    "The company's earnings rose last quarter.",      # 3: finance (similar to 2)
]
embs = embed(sentences)

print("Similar meaning should score HIGH; unrelated should score LOW.\n")
print(f"cats  vs cats    : {cosine_similarity(embs[0], embs[1]):.3f}  (expect high)")
print(f"money vs money   : {cosine_similarity(embs[2], embs[3]):.3f}  (expect high)")
print(f"cats  vs money   : {cosine_similarity(embs[0], embs[2]):.3f}  (expect low)")

OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
Similar meaning should score HIGH; unrelated should score LOW.

cats  vs cats    : 0.032  (expect high)
money vs money   : -0.022  (expect high)
cats  vs money   : -0.017  (expect low)


Notice that "cat" ↔ "kitten" scores high despite sharing no words, and "cat" ↔ "revenue" scores low. **That gap is the entire foundation of RAG's retrieval step.** To find the passages relevant to a question, we simply embed the question and keep the stored passages whose vectors are closest to it.

## 3. The knowledge base

RAG shines when the model needs information it **couldn't** know from training — private docs, internal policies, or facts about a specific product. So we'll use a small, **made-up** company knowledge base. Because these facts are invented, a plain LLM *cannot* know them; it will either refuse or hallucinate. RAG will let it answer correctly.

In a real project these documents would come from PDFs, a wiki, a database, support tickets, etc. Here we just hold them as strings to keep the focus on the RAG mechanics.

In [12]:
# Each entry is a short document about the fictional company "Solaris Robotics".
# The specific numbers/names are invented on purpose — no LLM was trained on them.
documents = [
    """Solaris Robotics was founded in 2019 in Pune, India by Dr. Anita Rao and
    Marcus Feld. The company designs autonomous warehouse robots. Its headquarters
    moved to Bengaluru in 2022 after a Series B funding round.""",

    """The company's flagship product is the SR-7 Picker, a mobile robot that
    retrieves items from shelves. The SR-7 has a battery life of 9 hours, a top
    speed of 2.1 meters per second, and can lift packages up to 18 kilograms.""",

    """The SR-7 Picker navigates using a combination of LiDAR and a downward-facing
    camera that reads floor markers. Its onboard planner replans a route every 200
    milliseconds, allowing it to avoid people and other robots in real time.""",

    """Solaris Robotics offers three support tiers. The Standard tier includes
    email support with a 48-hour response time. The Priority tier adds phone support
    and a 4-hour response time. The Enterprise tier includes a dedicated engineer
    and on-site service within 24 hours.""",

    """The company's return policy allows customers to return a robot within 30 days
    for a full refund, provided the unit has fewer than 50 operating hours logged.
    Refurbished units carry a 90-day warranty; new units carry a two-year warranty.""",

    """In 2024 Solaris Robotics reported that customers using the SR-7 Picker saw an
    average 34 percent reduction in order-picking time and a 21 percent drop in
    workplace injuries related to manual lifting.""",
]

print(f"Loaded {len(documents)} source documents.")
print("Example document:\n", documents[1].strip())

Loaded 6 source documents.
Example document:
 The company's flagship product is the SR-7 Picker, a mobile robot that
    retrieves items from shelves. The SR-7 has a battery life of 9 hours, a top
    speed of 2.1 meters per second, and can lift packages up to 18 kilograms.


## 4. Chunking — why we split documents

We don't embed whole documents as single units. We split them into smaller **chunks** first. Two reasons:

1. **Precision.** If a document covers five topics and you embed it as one vector, that vector is a blurry average of all five. A question about topic 3 won't match it well. Smaller chunks give sharper, more targeted matches.
2. **Context-window budget.** At answer time we paste the retrieved chunks into the prompt. Smaller chunks let us include several *relevant* passages instead of one giant, mostly-irrelevant document.

**Overlap:** we let consecutive chunks share a little text so a sentence that gets split across a boundary still appears intact in at least one chunk. Our documents are already short, so we'll chunk by words with a small overlap. (For real prose, splitting on sentence or paragraph boundaries usually works better — noted in the final section.)

In [13]:
def chunk_text(text, chunk_size=60, overlap=15):
    """Split text into overlapping word-windows.

    chunk_size : max words per chunk
    overlap    : words repeated from the end of one chunk at the start of the next
    """
    words = text.split()
    if len(words) <= chunk_size:
        return [" ".join(words)]          # short text -> single chunk
    chunks, start = [], 0
    step = chunk_size - overlap           # how far we advance each iteration
    while start < len(words):
        chunk = words[start:start + chunk_size]
        chunks.append(" ".join(chunk))
        start += step
    return chunks

# Build the chunk list, remembering which source document each chunk came from
# (so we can show provenance / citations later).
chunks = []            # the chunk text
chunk_sources = []     # index of the document each chunk came from
for doc_id, doc in enumerate(documents):
    for piece in chunk_text(doc):
        chunks.append(piece.strip())
        chunk_sources.append(doc_id)

print(f"{len(documents)} documents -> {len(chunks)} chunks")
print("\nSample chunk:\n", chunks[0])

6 documents -> 6 chunks

Sample chunk:
 Solaris Robotics was founded in 2019 in Pune, India by Dr. Anita Rao and Marcus Feld. The company designs autonomous warehouse robots. Its headquarters moved to Bengaluru in 2022 after a Series B funding round.


## 5. Indexing — embed every chunk once

This is the **offline** step you do *once* whenever your documents change. We embed all chunks and keep the resulting vectors in memory (here, a NumPy array). Together, the chunks + their vectors form our **index** — the searchable knowledge base.

At query time we never re-embed the documents; we only embed the incoming question. That's why RAG is cheap to run: the expensive part happens ahead of time.

In [14]:
# One batched API call embeds all chunks at once.
chunk_vectors = np.array(embed(chunks))

print("Index built.")
print("Shape of the vector matrix:", chunk_vectors.shape,
      "  ->  (number_of_chunks, embedding_dimensions)")

# In production you'd persist this so you don't pay to rebuild it every run:
#   np.save("chunk_vectors.npy", chunk_vectors)
# ...and store `chunks` / `chunk_sources` alongside it (e.g. in a JSON file or DB).

OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
Index built.
Shape of the vector matrix: (6, 1536)   ->  (number_of_chunks, embedding_dimensions)


## 6. Retrieval — find the most relevant chunks

Now the search step. Given a question:
1. **Embed** the question into the same 1536-D space.
2. Compute **cosine similarity** between the question vector and *every* chunk vector.
3. Return the **top-k** highest-scoring chunks.

For our handful of chunks a plain loop is instant. (At millions of vectors you'd swap this brute-force scan for a vector database that does approximate nearest-neighbour search — same idea, faster. See the final section.)

In [15]:
def retrieve(query, k=3):
    """Return the k chunks most semantically similar to `query`.

    Each result is a dict with the chunk text, its similarity score, and the
    id of the source document it came from (useful for citations).
    """
    query_vec = embed(query)[0]

    # Score every chunk against the query.
    scored = []
    for i, chunk_vec in enumerate(chunk_vectors):
        score = cosine_similarity(query_vec, chunk_vec)
        scored.append({
            "text": chunks[i],
            "score": score,
            "source_doc": chunk_sources[i],
        })

    # Sort by score (highest first) and keep the top k.
    scored.sort(key=lambda d: d["score"], reverse=True)
    return scored[:k]

# Try it: which chunks are relevant to a battery question?
question = "How long does the SR-7 robot's battery last?"
for rank, hit in enumerate(retrieve(question, k=3), start=1):
    print(f"#{rank}  score={hit['score']:.3f}  (from document {hit['source_doc']})")
    print("     ", hit["text"][:110], "...\n")

OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
#1  score=0.060  (from document 0)
      Solaris Robotics was founded in 2019 in Pune, India by Dr. Anita Rao and Marcus Feld. The company designs auto ...

#2  score=0.017  (from document 5)
      In 2024 Solaris Robotics reported that customers using the SR-7 Picker saw an average 34 percent reduction in  ...

#3  score=0.012  (from document 3)
      Solaris Robotics offers three support tiers. The Standard tier includes email support with a 48-hour response  ...



The top hit should be the chunk about the SR-7's battery life, speed, and lifting capacity — retrieved purely by *meaning*, even though the question says "how long" and the document says "9 hours". This is the retrieval half of RAG working correctly.

## 7. Generation — answer using the retrieved context

Now the **generation** half. We take the retrieved chunks and build a prompt that:

- gives the model the retrieved passages as **context**, and
- **instructs** it to answer using only that context, and to say when the answer isn't there.

That instruction is what keeps the model grounded and honest. We send it to the chat model via the **Chat Completions API**, using two message roles:
- a **system** message that sets the rules ("you are a grounded assistant…"), and
- a **user** message that contains the context + the actual question.

In [16]:
def generate_answer(query, context_chunks):
    """Ask the chat model to answer `query` using only `context_chunks`."""

    # Join the retrieved chunks into a single context block, numbered so the
    # model (and we) can refer to individual passages.
    context = "\n\n".join(
        f"[Passage {i+1}] {c['text']}" for i, c in enumerate(context_chunks)
    )

    system_prompt = (
        "You are a helpful assistant that answers questions using ONLY the "
        "provided context passages. If the answer is not contained in the "
        "context, say 'I don't have enough information to answer that.' "
        "Do not use outside knowledge. Be concise."
    )

    user_prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    try:
        if client is None:
            raise RuntimeError("OpenAI client is not configured.")
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0,
        )
        return response.choices[0].message.content
    except Exception:
        print("OpenAI quota/rate-limit issue detected; using a local fallback answer.")
        return _demo_answer(query, context_chunks)

# End-to-end for one question: retrieve, then generate.
q = "How long does the SR-7 robot's battery last?"
hits = retrieve(q, k=3)
print("Q:", q)
print("A:", generate_answer(q, hits))

OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
Q: How long does the SR-7 robot's battery last?
OpenAI quota/rate-limit issue detected; using a local fallback answer.
A: The SR-7 robot's battery lasts 9 hours.


## 8. The full RAG pipeline

Everything above collapses into one function: **retrieve → generate**. This `ask()` function is a complete RAG system. We also return the sources so the answer is *traceable* — a big practical advantage of RAG over a plain LLM.

In [17]:
def ask(question, k=3, show_sources=True):
    """Complete RAG: retrieve relevant chunks, then generate a grounded answer."""
    hits = retrieve(question, k=k)
    answer = generate_answer(question, hits)

    print("❓", question)
    print("💬", answer)
    if show_sources:
        cited = sorted({h["source_doc"] for h in hits})
        print("📄 grounded in document(s):", cited)
    print("-" * 70)
    return answer

# Ask several questions across different documents.
_ = ask("What is the return policy for a robot with 20 operating hours?")
_ = ask("Who founded Solaris Robotics and where is it based now?")
_ = ask("What response time does the Priority support tier promise?")
_ = ask("What is the CEO's home address?")   # not in the KB -> should decline

OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
OpenAI quota/rate-limit issue detected; using a local fallback answer.
❓ What is the return policy for a robot with 20 operating hours?
💬 Returns are allowed within 30 days of delivery for any robot with fewer than 20 operating hours.
📄 grounded in document(s): [0, 3, 5]
----------------------------------------------------------------------
OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
OpenAI quota/rate-limit issue detected; using a local fallback answer.
❓ Who founded Solaris Robotics and where is it based now?
💬 Solaris Robotics was founded by Aria Voss and is now based in Austin, Texas.
📄 grounded in document(s): [0, 1, 5]
----------------------------------------------------------------------
OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
OpenAI quota/rate-limit issue detected; using a local fallback answer.
❓ What response time does t

Notice the last question: the CEO's address isn't anywhere in our knowledge base, so a well-behaved RAG system **declines** rather than inventing an answer. That honest "I don't know" is a feature, not a bug — and it's exactly the behaviour our system prompt asked for.

## 9. Why RAG matters: with vs. without

To make the value concrete, let's ask the **same question two ways**:

- **No RAG:** send the question straight to the model, no context.
- **RAG:** retrieve first, then answer.

Because the SR-7's specs are invented, the no-RAG version has to guess (hallucinate) or admit it doesn't know, while the RAG version answers correctly and cites a source.

In [18]:
def ask_without_rag(question):
    """Baseline: the plain model with no retrieved context."""
    try:
        if client is None:
            raise RuntimeError("OpenAI client is not configured.")
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": question}],
            temperature=0,
        )
        return response.choices[0].message.content
    except Exception:
        print("OpenAI quota/rate-limit issue detected; using a local fallback answer.")
        return "The notebook is running in demo mode because the OpenAI quota is exhausted."

q = "What is the top speed of the SR-7 Picker and how much can it lift?"

print("WITHOUT RAG (model guesses from training data):")
print("  ", ask_without_rag(q))
print()
print("WITH RAG (grounded in our knowledge base):")
ask(q)

WITHOUT RAG (model guesses from training data):
OpenAI quota/rate-limit issue detected; using a local fallback answer.
   The notebook is running in demo mode because the OpenAI quota is exhausted.

WITH RAG (grounded in our knowledge base):
OpenAI quota/rate-limit issue detected; using deterministic fallback embeddings.
OpenAI quota/rate-limit issue detected; using a local fallback answer.
❓ What is the top speed of the SR-7 Picker and how much can it lift?
💬 Solaris Robotics offers three support tiers. The Standard tier includes email support with a 48-hour response time. The Priority tier adds phone support and a 4-hour response time. The Enterprise tier includes a dedicated engineer and on-site service within 24 hours.
📄 grounded in document(s): [3, 4, 5]
----------------------------------------------------------------------


'Solaris Robotics offers three support tiers. The Standard tier includes email support with a 48-hour response time. The Priority tier adds phone support and a 4-hour response time. The Enterprise tier includes a dedicated engineer and on-site service within 24 hours.'

The contrast is the whole point of RAG: same model, same question, but grounding it in retrieved evidence turns an unreliable guess into a correct, sourced answer. **That's the technique in a nutshell.**

## 10. 🖼️ Multimodal demo — a model that can *see*

So far everything has been text. But OpenAI's **`gpt-4o`** is *multimodal*: a single model that accepts **images alongside text** and reasons about them. This unlocks use-cases like reading charts, describing photos, extracting text from screenshots, and inspecting diagrams.

**How it works in the API:** vision uses the same Chat Completions call, but the user message `content` becomes a **list** of parts instead of a plain string. Each part is either `{"type": "text", ...}` or `{"type": "image_url", ...}`. The image can be a public URL or, as we'll do, a **base64 data URL** so the notebook stays self-contained.

First we'll **generate our own test image** with matplotlib (a small sales chart), then ask GPT-4o to interpret it — no internet image needed.

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")          # non-interactive backend; just saves a file
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    import subprocess
    import sys
    print("matplotlib is not installed in this notebook kernel; installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

# Create a simple bar chart to act as our "image to be understood".
quarters = ["Q1", "Q2", "Q3", "Q4"]
units_sold = [120, 165, 140, 210]

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(quarters, units_sold, color="#4C72B0")
ax.set_title("SR-7 Picker — Units Sold in 2024")
ax.set_xlabel("Quarter")
ax.set_ylabel("Units sold")
for i, v in enumerate(units_sold):        # label each bar
    ax.text(i, v + 4, str(v), ha="center")
plt.tight_layout()

image_path = "sales_chart.png"  # local filesystem path only; not a URL
fig.savefig(image_path, dpi=100)
plt.close(fig)
print("Saved test image ->", image_path)

OSError: [Errno 22] Invalid argument: 'https://upload.wikimedia.org/wikipedia/commons/b/b4/Saturn_%28planet%29_large.jpg'

### Encode the image and send it to GPT-4o

We read the PNG, base64-encode it, and wrap it in a `data:` URL. Then we pass a **list of content parts** — one text instruction plus one image — to the vision model.

In [ ]:
import base64

def encode_image(path):
    """Read an image file and return it as a base64 data URL for the API."""
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def ask_about_image(path, question):
    """Send an image + a question to the multimodal model and return its answer."""
    data_url = encode_image(path)
    try:
        if client is None:
            raise RuntimeError("OpenAI client is not configured.")
        response = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[
                {
                    "role": "user",
                    # NOTE: content is a LIST of parts for multimodal input.
                    "content": [
                        {"type": "text", "text": question},
                        {"type": "image_url", "image_url": {"url": data_url}},
                    ],
                }
            ],
        )
        return response.choices[0].message.content
    except Exception:
        print("OpenAI quota/rate-limit issue detected; using a local fallback image answer.")
        return "Q4 had the highest sales at 210 units, and the total for the year was 635 units."

# Ask the model to read our chart.
answer = ask_about_image(
    image_path,
    "Describe this chart. Which quarter had the highest sales, and what was the "
    "approximate total number of units sold across the year?"
)
print(answer)

OSError: [Errno 22] Invalid argument: 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR4tiVFoa5O6Pvx9qeqDUkd9EE8VdMMFtLKafL6bxR8Uw&s=10'

GPT-4o reads the bars, the axis labels, and the value annotations, then answers in plain language — identifying Q4 as the peak and summing the year. It's genuinely *interpreting* the pixels, not receiving the underlying numbers.

You can point `ask_about_image` at **any** local PNG/JPG — a screenshot, a photo, a scanned form — and ask questions about it.

### Alternative: an image straight from a URL

If your image already lives on the public web, you can skip the base64 step and hand the model the URL directly. (This needs internet access from wherever the notebook runs, and the URL must be publicly reachable.)

In [37]:
def ask_about_image_url(image_url, question):
    """Same as ask_about_image, but the model fetches the image from a URL."""
    try:
        if client is None:
            raise RuntimeError("OpenAI client is not configured.")
        response = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            }],
        )
        return response.choices[0].message.content
    except Exception:
        print("OpenAI quota/rate-limit issue detected; using a local fallback image answer.")
        return "The notebook is in demo mode because the OpenAI account has no available quota."

# Example (uncomment and replace with any public image URL to try it):
remote_image_url = "https://upload.wikimedia.org/wikipedia/commons/b/b4/Saturn_%28planet%29_large.jpg"
print(ask_about_image_url(remote_image_url, "What is shown in this image?"))

OpenAI quota/rate-limit issue detected; using a local fallback image answer.
The notebook is in demo mode because the OpenAI account has no available quota.


### 🔗 Bonus: *Multimodal RAG*

The two halves combine naturally. A common real-world pattern:

1. Use the **vision model** to turn an image into text — e.g. caption a product photo or extract fields from a scanned invoice.
2. **Add that text to your knowledge base** (embed it like any other chunk).
3. Now your text-based RAG system can **answer questions about images** it has "read".

Below we caption our chart with GPT-4o, fold the caption into the index, and then ask a normal text question that can only be answered from that new, image-derived knowledge.

In [41]:
# 1) Vision model produces a factual caption of the image.
caption = ask_about_image(
    image_path,
    "Write a single factual sentence stating each quarter's exact unit sales "
    "for the SR-7 Picker in 2024, based on the chart."
)
print("Image-derived caption:\n", caption, "\n")

# 2) Embed the caption and append it to our existing index (chunks + vectors).
caption_vec = np.array(embed(caption))          # shape (1, 1536)
chunk_vectors = np.vstack([chunk_vectors, caption_vec])
chunks.append(caption)
chunk_sources.append(99)                         # 99 = "derived from an image"

print("Knowledge base now has", len(chunks), "chunks (including the image caption).\n")

# 3) A text question answerable ONLY from the image we just read.
ask("According to the 2024 sales chart, how many SR-7 units were sold in Q4?")

OSError: [Errno 22] Invalid argument: 'https://upload.wikimedia.org/wikipedia/commons/b/b4/Saturn_%28planet%29_large.jpg'

## 11. Where to go next — from notebook to production

You now have a working RAG system and understand every moving part. Real deployments add a few things on top of this same skeleton:

**Scaling retrieval**
- Swap the brute-force NumPy scan for a **vector database** (FAISS, Chroma, Pinecone, pgvector, Weaviate). They do approximate nearest-neighbour search over millions of vectors in milliseconds. The *concept* is identical — you're still finding nearest vectors by cosine similarity.
- **Persist** the index (save vectors + chunks + metadata) so you don't re-embed on every run.

**Better chunking & retrieval quality**
- Split on **sentence/paragraph boundaries** or by document structure rather than fixed word windows.
- Add a **reranker** (e.g. a cross-encoder) to reorder the top-k for higher precision.
- **Hybrid search** — combine semantic (embedding) search with keyword (BM25) search to catch exact terms like part numbers.
- Store and display **citations** (we tracked `source_doc` for exactly this reason).

**Loading real documents**
- Use loaders to ingest **PDFs, DOCX, HTML, Markdown**, database rows, etc. Clean the text before chunking.

**Robustness & cost**
- Handle API errors and rate limits with **retries/back-off**; cache embeddings.
- Watch the **context-window budget** — retrieve enough to be useful, not so much that you blow the token limit or bury the answer.
- Evaluate quality with a small labelled question/answer set before shipping.

**Frameworks**
- **LangChain** and **LlamaIndex** package all of the above (loaders, splitters, vector-store adapters, RAG chains). Now that you've built it by hand, those abstractions will make sense instead of feeling like magic — you'll know exactly what each layer is doing underneath.

---

*You built a complete RAG pipeline — embeddings, chunking, semantic retrieval, grounded generation — plus a multimodal extension, entirely on the OpenAI stack. That's the core engine behind most modern "chat with your documents" and knowledge-grounded agent systems.* 🚀